# Multimodal sexism identification and characterization in memes

**Serrano Team submission to EXIST 2026 Task 2**

This is the recommended entry point to the repository. The project implements a hierarchical multimodal system for three EXIST 2026 meme tasks: binary sexism identification, source-intention classification, and multilabel sexism-facet classification.

![Pipeline](../assets/figures/final_pipeline_overview.png)

The best official soft-soft runs rank **#14/139**, **#10/112**, and **#11/113** for Tasks 2.1–2.3. These are run-level all-language positions from the official EXIST 2026 overview. Internal experiment values below come from the final audited report in `../report/serrano-team-exist2026-report.pdf`; the two evidence scopes are separate.

## Dataset and task structure

The training collection contains **3,984 memes** and the test collection contains **1,053**. The training data include **2,005 English** and **1,979 Spanish** memes, together with OCR text, images, and physiological signals from eye tracking, heart rate, and EEG.

The system follows the task hierarchy:

1. **Task 2.1:** route each meme to `NO` or `YES`.
2. **Task 2.2:** for routed sexist memes, predict `DIRECT` or `JUDGEMENTAL`.
3. **Task 2.3:** for routed sexist memes, predict one or more of five sexism facets.

## Report-aligned validation results

| Component | Evaluation scope | Selected result |
|---|---|---:|
| Task 2.1 gate | 674-example development/model-selection split | **0.709 macro-F1** |
| Task 2.1 gate | `YES` class | **0.815 F1**, **0.94 recall** |
| Task 2.1 gate | Overall | **0.748 accuracy** |
| Task 2.2 ensemble | 357 sexist-only development/model-selection examples | **0.637 macro-F1** |
| Task 2.2 ensemble | `JUDGEMENTAL` class | **0.469 F1** |
| Task 2.2 cascade | Three-class routing diagnostic | **0.485 macro-F1** |
| Task 2.3 ensemble | 398 facet-positive development/model-selection examples | **0.677 macro-F1** |
| Task 2.3 ensemble | Multilabel | **0.700 micro-F1**, **0.694 samples-F1** |
| Task 2.3 + gate | Positive-only routing diagnostic | **0.674 macro-F1** |

These scopes are intentionally separated. Conditional classifiers should not be compared directly with their routed cascades.

## Why the final design works

The strongest result did not come from one universally superior model. It came from making each design choice answer a specific failure mode:

- **Recall-oriented routing:** a false `NO` is irreversible, so Task 2.1 uses a threshold of 0.371642 and accepts more false positives to preserve sexist examples for downstream classification.
- **Error diversity:** E5, CLIP, DINOv2, boosted trees, and language specialists fail differently; equal-weight voting reduces dependence on one representation.
- **Pragmatic enrichment:** Task 2.2 must distinguish endorsement from criticism, so VLM descriptions make text-image relationships explicit while OCR-only models remain as robustness anchors.
- **Stability before score chasing:** Task 2.2 keeps only candidates with development macro-F1 of at least 0.50 and an absolute calibration-development gap no larger than 0.08.
- **Label-specific decisions:** Task 2.3 tunes one threshold per facet and includes a classifier chain to capture recurring co-occurrence.
- **Selective physiological fusion:** sensors are retained only where a paired ablation improves the same content branch.

This is the project's main experimental lesson: multimodality helps when each modality has a defined role and a controlled ablation, not when every feature is concatenated by default.

## Final submission audit

These counts are completeness and distribution checks from the report, not test-set performance scores.

| Task | Final hard-run distribution over 1,053 memes |
|---|---|
| Task 2.1 | 608 `YES`, 445 `NO` |
| Task 2.2 | 445 `NO`; among routed positives: 396 `DIRECT`, 212 `JUDGEMENTAL` |
| Task 2.3 | 445 `NO`; facet counts: 420 stereotyping/dominance, 293 ideological inequality, 273 objectification, 270 misogyny/non-sexual violence, 194 sexual violence |

The curated Task 2.2 and Task 2.3 export cells validate against these report values and reject superseded intermediate distributions.

## Notebook path

1. `01_eda_and_sensor_analysis.ipynb` — data, OCR, visual, uncertainty, and physiological-signal audit.
2. `02_task21_binary_gate.ipynb` — eight-expert multimodal `NO/YES` gate.
3. `03_task22_source_intention.ipynb` — robust candidate benchmark and five-member geometric ensemble.
4. `04_task23_sexism_facets.ipynb` — eight-expert multilabel facet ensemble and gate diagnostic.

For a compact methodological account, continue with `../docs/METHODOLOGY.md`. For exact environment and cache expectations, see `../docs/REPRODUCIBILITY.md`.